In [3]:
library(tidyverse)
library(surveillance)
library(here) 

dataset_path <- here("data", "interim", "processed_fires_dataset.csv")

# load dataset and convert date to numeric offset
fire_df <- read_csv(dataset_path, show_col_types = FALSE) %>%
  mutate(time_days = as.numeric(as.Date(acq_date) - min(as.Date(acq_date))))

coords <- as.matrix(fire_df[, c("longitude", "latitude")])

# precompute distance matrices
dist_s <- dist(coords)
dist_t <- dist(fire_df$time_days)
n_pairs <- as.numeric(length(dist_s))

# safe helper function ensuring scalar outputs
evaluate_knox_safe <- function(ds_val, dt_val) {
  res <- tryCatch(
    knox(
      dt = dist_t,
      ds = dist_s,
      eps.t = dt_val,
      eps.s = ds_val,
      simulate.p.value = FALSE
    ),
    error = function(e) NULL
  )
  
  if (is.null(res)) {
    return(tibble(
      `d_s (space)` = as.numeric(ds_val),
      `d_t (time)` = as.numeric(dt_val),
      `Observed (X)` = NA_real_,
      `Expected (E[X])` = NA_real_,
      `Knox Ratio` = NA_real_,
      `p-value` = "Error",
      `Status` = "FAILED"
    ))
  }
  
  obs <- as.numeric(res$statistic)[1]
  close_s <- as.numeric(sum(dist_s <= ds_val))[1]
  close_t <- as.numeric(sum(dist_t <= dt_val))[1]
  
  exp_val <- (close_s * close_t) / n_pairs
  ratio <- if (exp_val > 0) obs / exp_val else NA_real_
  p_val <- as.numeric(res$p.value)[1]
  
  tibble(
    `d_s (space)` = as.numeric(ds_val),
    `d_t (time)` = as.numeric(dt_val),
    `Observed (X)` = obs,
    `Expected (E[X])` = round(exp_val, 1),
    `Knox Ratio` = round(ratio, 4),
    `p-value` = ifelse(!is.na(p_val) && p_val < 0.001, "< 0.001", sprintf("%.3f", p_val)),
    `Status` = ifelse(!is.na(ratio) && ratio >= 0.90 && ratio <= 1.10, "PASSED", "FAILED")
  )
}

# grid of spatial and temporal parameters
grid_params <- expand.grid(
  ds = c(10.0, 12.0, 15.0),
  dt = c(7.0, 15.0, 30.0)
)

# run evaluation safely across parameters
results_df <- map2_dfr(grid_params$ds, grid_params$dt, evaluate_knox_safe)

# print plain dataframe output
as.data.frame(results_df)

d_s (space),d_t (time),Observed (X),Expected (E[X]),Knox Ratio,p-value,Status
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
10,7,1877,1381.8,1.3583,< 0.001,FAILED
12,7,2154,1709.8,1.2598,< 0.001,FAILED
15,7,2524,2177.6,1.1591,< 0.001,FAILED
10,15,2874,2480.6,1.1586,< 0.001,FAILED
12,15,3419,3069.4,1.1139,< 0.001,FAILED
15,15,4136,3909.0,1.0581,< 0.001,PASSED
10,30,4688,4358.4,1.0756,< 0.001,PASSED
12,30,5646,5392.9,1.0469,< 0.001,PASSED
15,30,7011,6868.2,1.0208,0.042,PASSED
